In [4]:
import os
import re
import glob
import numpy as np
import pandas as pd
from scipy.ndimage import distance_transform_edt
from tqdm import tqdm

In [5]:
# --- НАСТРОЙКИ ---
RAW_ROOT = '/app/urban-layer/'
SAVE_ROOT = '/app/urban-layer-dataset/'
GRID_DIMS = {'x': 128, 'y': 128, 'z': 32} 
TARGET_Z_IDX = 1

In [6]:
# Словарь для маппинга осей Tecplot в индексы Numpy (Z, Y, X)
# В файлах порядок X, Y, Z. Numpy reshape делает (Z, Y, X).
# X меняется быстрее всего (внутренний цикл), Z медленнее всего.

class RobustConfigParser:
    def __init__(self, filepath):
        self.filepath = filepath
        with open(filepath, 'r') as f:
            self.lines = [line.strip() for line in f if line.strip() and not line.strip().startswith('#')]
        
        self.domain_params = self._parse_simple_block('domain')
        self.grid_params = self._parse_simple_block('grid')
        self.wind_params = self._extract_wind_forcing()
        self.tracers = self._parse_tracers_robust()

        # Вычисляем скейлинг (метры -> индексы)
        # Если параметров нет, фоллбэк на дефолты из твоего конфига
        len_x = self.domain_params.get('length', 256.0)
        len_y = self.domain_params.get('width', 256.0)
        len_z = self.domain_params.get('height', 64.0)
        
        self.scale_x = self.grid_params.get('cx', 128) / len_x
        self.scale_y = self.grid_params.get('cy', 128) / len_y
        self.scale_z = self.grid_params.get('cz', 32)  / len_z

    def _extract_value(self, text):
        """Пытается извлечь число из строки вида 'key = 123.45;'"""
        # Ищем число после знака равно
        match = re.search(r'=\s*([-\d.eE]+)', text)
        if match:
            return float(match.group(1))
        return 0.0

    def _extract_value_for_emission(self, text, key):
        """
        Пытается извлечь число для конкретного ключа.
        Ищет паттерн: граница_слова + ключ + пробелы + равно + число
        """
        # Ищем конкретный ключ, за которым следует равно и число.
        # \b означает границу слова (чтобы поиск 'min' не нашел 'xmin')
        pattern = rf'\b{key}\s*=\s*([-\d.eE]+)'
        
        match = re.search(pattern, text)
        if match:
            return float(match.group(1))
        return None  # Лучше возвращать None, если не найдено, чем 0.0

    def _extract_wind_forcing(self):
        """Ищем dPdx и dPdy по всему файлу"""
        wind = {'dPdx': 0.0, 'dPdy': 0.0}
        for line in self.lines:
            if 'dPdx' in line:
                wind['dPdx'] = self._extract_value(line)
            if 'dPdy' in line:
                wind['dPdy'] = self._extract_value(line)
        return wind

    def _parse_simple_block(self, block_name):
        """Парсит одноуровневый блок (domain, grid)"""
        params = {}
        in_block = False
        for line in self.lines:
            if line.startswith(block_name) and '{' in line:
                in_block = True
                continue
            if in_block:
                if '}' in line:
                    break
                # Парсим ключи x = 0.0; y = 0.0;
                parts = line.split(';')
                for part in parts:
                    if '=' in part:
                        k, v = part.split('=')
                        try:
                            params[k.strip()] = float(re.search(r'([-\d.eE]+)', v).group(1))
                        except: pass
        return params

    def _parse_tracers_robust(self):
        """
        Парсер, который учитывает вложенность скобок для трейсеров.
        Возвращает {tracer_id: [source_box_dict, ...]}
        """
        tracers = {}
        current_tracer = None
        current_content = []
        brace_count = 0
        in_tracer_block = False

        for line in self.lines:
            # Начало блока tracer_N
            match_start = re.match(r'tracer_(\d+)\s*\{', line)
            if match_start:
                current_tracer = int(match_start.group(1))
                in_tracer_block = True
                brace_count = 1
                current_content = []
                continue

            if in_tracer_block:
                brace_count += line.count('{')
                brace_count -= line.count('}')
                current_content.append(line)

                # Конец блока tracer
                if brace_count == 0:
                    # Парсим содержимое блока tracer на предмет point_emission
                    tracers[current_tracer] = self._extract_emissions_from_text(current_content)
                    in_tracer_block = False
                    current_tracer = None
        
        return tracers

    def _extract_emissions_from_text(self, lines):
        """Ищет point_emission блоки внутри текста одного трейсера"""
        sources = []
        in_emission = False
        current_source = {}
        
        for line in lines:
            if 'point_emission_' in line:
                in_emission = True
                current_source = {}
                continue
            
            if in_emission:
                if '}' in line:
                    if current_source: sources.append(current_source)
                    in_emission = False
                    continue
                
                # Парсим координаты
                for axis in ['xmin', 'xmax', 'ymin', 'ymax']:
                    # Передаем axis в функцию
                    val = self._extract_value_for_emission(line, axis)
                    
                    # Если значение найдено, записываем его
                    if val is not None:
                        current_source[axis] = val
                current_source['zmin'] = 0.0
                current_source['zmax'] = 8.0

        return sources

    def get_source_mask(self, tracer_id, shape=(32, 128, 128)):
        mask = np.zeros(shape, dtype=np.float32)
        sources = self.tracers.get(tracer_id, [])
        
        if not sources:
            print(f"WARNING: No sources found for tracer {tracer_id}")
            return mask

        for box in sources:
            try:
                # Переводим физические координаты в индексы массива
                x1 = int(box.get('xmin', 0) * self.scale_x)
                x2 = int(box.get('xmax', 0) * self.scale_x)
                y1 = int(box.get('ymin', 0) * self.scale_y)
                y2 = int(box.get('ymax', 0) * self.scale_y)
                z1 = int(box.get('zmin', 0) * self.scale_z)
                z2 = int(box.get('zmax', 0) * self.scale_z)

                # Ограничиваем границами
                x1, x2 = max(0, x1), min(shape[2], x2)
                y1, y2 = max(0, y1), min(shape[1], y2)
                z1, z2 = max(0, z1), min(shape[0], z2)

                mask[z1:z2, y1:y2, x1:x2] = 1.0
            except Exception as e:
                print(f"Error making mask for tracer {tracer_id}: {e}")
        
        return mask

In [7]:
def read_tecplot_robust(filepath, expected_shape=(32, 128, 128)):
    """
    Читает ASCII .plt файл, динамически определяя начало данных.
    """
    header_rows = 0
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            line = line.strip()
            # Данные начинаются с числа (например, 1.000) или минуса
            if line and (line[0].isdigit() or line[0] == '-'):
                header_rows = i
                break
    
    try:
        # engine='c' значительно быстрее для больших файлов
        df = pd.read_csv(filepath, sep=r'\s+', skiprows=header_rows, 
                         names=['x', 'y', 'z', 'val'], engine='python')
        
        vals = df['val'].values
        
        # Проверка целостности
        expected_len = np.prod(expected_shape)
        if len(vals) != expected_len:
            print(f"Error {filepath}: Expected {expected_len} points, got {len(vals)}")
            return None

        # Reshape: Tecplot point format (Outer Loop K(Z), Middle J(Y), Inner I(X))
        # Numpy reshape заполняет сначала последний индекс -> (K, J, I) == (Z, Y, X)
        tensor = vals.reshape(expected_shape)
        return tensor
        
    except Exception as e:
        print(f"Read error {filepath}: {e}")
        return None

In [8]:
def extract_number(filename):
    # Извлекаем число N из C[N]-avg-.plt
    match = re.search(r'C\[(\d+)\]-avg-\.plt', filename)
    return int(match.group(1)) if match else float('inf') 

def sort_filenames(filenames):
    return sorted(filenames, key=extract_number)

def get_filenames(file_path, file_pattern):
    file_list = [f for f in os.listdir(file_path) 
                 if os.path.isfile(os.path.join(file_path, f)) and re.search(file_pattern, f)]
    file_list = sort_filenames(file_list)
    return file_list

In [9]:
def process_experiment(build_path, output_path):
    """Пайплайн обработки одного запуска"""
    
    build_name = os.path.basename(os.path.normpath(build_path))
    output_name = os.path.basename(os.path.normpath(output_path))
    
    config_file = os.path.join(output_path, 'config.txt')
    if not os.path.exists(config_file):
        print(f"Skipping {output_name}: No config.txt")
        return

    # 1. Парсим конфиг
    try:
        cfg = RobustConfigParser(config_file)
    except Exception as e:
        print(f"Config Parse Error in {output_name}: {e}")
        return

    # 2. Ищем и обрабатываем LAD (Деревья)
    # Путь: common/3d/LAD*.plt
    lad_files = glob.glob(os.path.join(output_path, 'common', '3d', 'LAD*.plt'))
    if not lad_files: return # Без деревьев нет смысла

    lad_tensor = read_tecplot_robust(lad_files[0])
    if lad_tensor is None: return

    # Создаем маски из LAD файла
    # -9999 -> Здание
    # > 0   -> Дерево
    buildings_mask = (lad_tensor < -9000).astype(np.float32)
    trees_density = lad_tensor.copy()
    trees_density[buildings_mask == 1] = 0.0 # Обнуляем плотность листвы внутри стен
    
    # SDF считаем от зданий (расстояние до стен)
    air_mask = 1.0 - buildings_mask
    sdf = distance_transform_edt(air_mask)
    # Нормируем на макс размер домена, чтобы числа были ~0..1
    sdf = sdf / 128.0 

    # 3. Ищем файлы концентраций
    # output_*/stat-3d/C[d+]-avg-.plt
    file_pattern = r'C\[\d+\]-avg-\.plt'
    conc_files = get_filenames(output_path + '/stat-3d/', file_pattern)
    # conc_files = glob.glob(os.path.join(output_path, 'stat-3d', 'C[*]-avg-*.plt'))
    
    for c_file in conc_files:
        try:
            tracer_id = int(re.search(r'C\[(\d+)\]', os.path.basename(output_path + '/stat-3d/' + c_file)).group(1)) + 1
        except: continue
        # Читаем полную 3D концентрацию (так как она нужна для извлечения среза)
        conc = read_tecplot_robust(output_path + '/stat-3d/' + c_file)
        if conc is None: continue
        
        conc[conc < -9000] = 0.0
        
        source_mask = cfg.get_source_mask(tracer_id)
        if source_mask.sum() == 0:
            pass

        # 4. Формируем входной тензор (ОСТАЕТСЯ 3D, так как физика объемная)
        # Channels: [SDF, Trees, Source, WindX, WindY]
        wind_x = np.full(conc.shape, cfg.wind_params['dPdx'], dtype=np.float32)
        wind_y = np.full(conc.shape, cfg.wind_params['dPdy'], dtype=np.float32)
        
        x_tensor = np.stack([sdf, trees_density, source_mask, wind_x, wind_y], axis=0).astype(np.float32)
        # x_tensor shape: (5, 32, 128, 128)

        # --- ИЗМЕНЕНИЕ: Формируем Y_TARGET (2D срез на высоте 3м) ---
        # TARGET_Z_METERS = 3.0
        
        # Вычисляем индекс сетки. scale_z = (cells / meters)
        # Например, 32 ячейки / 64 метра = 0.5 ячеек/м. 
        # 3.0м * 0.5 = 1.5 -> берем индекс 1 (второй слой, центр ~3м)
        target_z_idx = TARGET_Z_IDX
        
        # Clip на случай, если высота задана выше домена
        target_z_idx = np.clip(target_z_idx, 0, conc.shape[0] - 1)
        
        # Извлекаем 2D срез из 3D массива (Z, Y, X)
        conc_slice = conc[target_z_idx, :, :]
        
        # Добавляем измерение канала: (Y, X) -> (1, Y, X)
        y_tensor = np.expand_dims(conc_slice, axis=0).astype(np.float32)
        # y_tensor shape: (1, 128, 128)
        # Сохраняем
        save_fname = f"{build_name}__{output_name}__tracer{tracer_id}.npz"
        np.savez_compressed(os.path.join(SAVE_ROOT, save_fname), x=x_tensor, y=y_tensor)

In [10]:
os.makedirs(SAVE_ROOT, exist_ok=True)
    
# Рекурсивный поиск всех output папок
# Предполагаем структуру /app/urban-layer/build*/output_*
# search_pattern = os.path.join(RAW_ROOT, 'build*', 'output_*')
search_pattern = os.path.join(RAW_ROOT, 'build_cuda1222_scale', 'output_*')

all_outputs = glob.glob(search_pattern)

all_outputs.sort(key=os.path.getctime)

print(f"Found {len(all_outputs)} output directories.")

for out_path in tqdm(all_outputs[:-1]):
    # Находим родительскую папку build для имени
    build_path = os.path.dirname(out_path)
    process_experiment(build_path, out_path)

Found 11 output directories.


100%|██████████| 10/10 [15:55<00:00, 95.53s/it]
